# 03 · QLoRA fine-tuning — phi4-mini (Kaggle T4×2)

**Dissertation Phase-3 headline (`[P2]`):** fine-tune a small model on the task's
**train split only**, then re-run the *same* frozen 20×4 grid for a clean
before→after delta — *"a fine-tuned 3.8B matches/beats a frontier LLM zero-shot on
this narrow KIE task at a fraction of cost/latency."*

> ⚠️ **DATA HYGIENE (non-negotiable).** The SFT data (`data/train/sft.jsonl`) is
> built from the **train split only** by `scripts/make_train_data.py`, which asserts
> zero test-set doc_ids before writing. The frozen 20-doc test set must never enter
> training. Do not hand-edit the data here.

**Runtime:** Kaggle → Settings → Accelerator **GPU T4 ×2**, Internet **On**.
Hyperparameters come from `config/train.yaml`. End result: a GGUF + Ollama Modelfile
you download and import locally with `scripts/import_finetuned.sh`.

## 0 · Environment

In [ ]:
# Unsloth brings its own pinned torch/transformers/trl/peft/bitsandbytes stack.
%pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

import torch
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU — set Accelerator to T4 x2")

## 1 · Repo + SFT data + config

Bring the repo (for `config/train.yaml`) and the SFT file. Two options:
- **git clone** the repo inside the notebook (needs Internet On), or
- **Add Data → upload** `data/train/sft.jsonl` (then set `SFT_PATH` to the Kaggle input).

In [ ]:
import os, json, yaml, glob
from pathlib import Path

# Option A: clone the repo (Phase-3 code lives on this branch, not main).
# NOTE: the GitHub repo is named `slm-vs-llm-kie`, and the project sits in a
# `slm-vs-llm-kie/` subdirectory inside it — PROJ resolves that below.
REPO = Path("/kaggle/working/slm-llm")
BRANCH = "feature/phase3-qlora-finetune"
if not REPO.exists():
    !git clone --depth 1 -b {BRANCH} https://github.com/Azhar-ali7/slm-vs-llm-kie.git {REPO} || echo 'clone skipped — using uploaded data'

PROJ = REPO / "slm-vs-llm-kie" if (REPO / "slm-vs-llm-kie").exists() else REPO

# Locate config/train.yaml (repo) and sft.jsonl (repo build or an uploaded Kaggle input).
TRAIN_CFG = PROJ / "config" / "train.yaml"
tcfg = yaml.safe_load(TRAIN_CFG.read_text()) if TRAIN_CFG.exists() else None

candidates = [PROJ / "data" / "train" / "sft.jsonl", *map(Path, glob.glob("/kaggle/input/**/sft.jsonl", recursive=True))]
SFT_PATH = next((p for p in candidates if p.exists()), None)
assert SFT_PATH, "sft.jsonl not found — clone the repo+build, or upload it as a Kaggle dataset."
print("config:", TRAIN_CFG if tcfg else "(defaults)", "| sft:", SFT_PATH)

# Fine-tune the first model in train.yaml (phi4-mini-ft).
MODEL_KEY = next(iter(tcfg["models"])) if tcfg else "phi4-mini-ft"
MCFG = (tcfg["models"][MODEL_KEY] if tcfg else
        {"base_repo": "unsloth/Phi-4-mini-instruct", "base_ollama": "phi4-mini", "ollama_tag": "phi4-mini-ft"})
LORA = (tcfg or {}).get("lora", {})
SFT  = (tcfg or {}).get("sft", {})
EXPORT = (tcfg or {}).get("export", {"quants": ["q4_k_m"], "out_dir": "artifacts"})
print("model:", MODEL_KEY, "->", MCFG)

## 2 · Load base model (4-bit) + attach LoRA

In [ ]:
from unsloth import FastLanguageModel

# 4096 fits 100% of Kleister docs (p99~3.1k tok, max~3.4k). 2048 truncated the
# response off ~41% of them -> labels all -100 -> silently dropped from training.
max_seq_len = int(SFT.get("max_seq_len", 4096))
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MCFG["base_repo"],
    max_seq_length = max_seq_len,
    load_in_4bit = bool(SFT.get("load_in_4bit", True)),
    dtype = None,   # auto (bf16/fp16)
)

model = FastLanguageModel.get_peft_model(
    model,
    r = int(LORA.get("r", 16)),
    lora_alpha = int(LORA.get("alpha", 16)),
    lora_dropout = float(LORA.get("dropout", 0.0)),
    bias = LORA.get("bias", "none"),
    target_modules = LORA.get("target_modules",
        ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]),
    use_gradient_checkpointing = LORA.get("use_gradient_checkpointing", "unsloth"),
    random_state = int(SFT.get("seed", 42)),
)

## 3 · Format the SFT data with the model's chat template

Each row is `{prompt, completion}` where `prompt` is the **exact eval-time prompt**
(from `build_prompt`). We wrap it as a user→assistant turn using the base model's
chat template, then train on the **completion only** (`train_on_responses_only`) so
the loss is on the JSON, not the instruction.

In [ ]:
from datasets import load_dataset as hf_load_dataset

raw = hf_load_dataset("json", data_files=str(SFT_PATH), split="train")
print("examples:", len(raw), "| fields:", raw.column_names)

def to_text(ex):
    msgs = [
        {"role": "user", "content": ex["prompt"]},
        {"role": "assistant", "content": ex["completion"]},
    ]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

ds = raw.map(to_text, remove_columns=raw.column_names)
print(ds[0]["text"][:600])

## 4 · Train (SFTTrainer)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import train_on_responses_only

args = TrainingArguments(
    per_device_train_batch_size = int(SFT.get("per_device_batch_size", 2)),
    gradient_accumulation_steps = int(SFT.get("grad_accum_steps", 4)),
    num_train_epochs = float(SFT.get("epochs", 3)),
    learning_rate = float(SFT.get("learning_rate", 2e-4)),
    lr_scheduler_type = SFT.get("lr_scheduler", "linear"),
    warmup_ratio = float(SFT.get("warmup_ratio", 0.03)),
    weight_decay = float(SFT.get("weight_decay", 0.01)),
    optim = SFT.get("optim", "adamw_8bit"),
    seed = int(SFT.get("seed", 42)),
    logging_steps = 10,
    output_dir = "outputs",
    # No checkpointing: a headless (papermill) run trips a TRL/Unsloth SFTConfig
    # class-identity PicklingError when the Trainer pickles training_args.bin. We
    # don't need checkpoints — the adapter is saved explicitly below via PEFT
    # save_pretrained (which does not pickle SFTConfig).
    save_strategy = "no",
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    report_to = "none",
    # Unsloth fused-loss + Transformers' new default average_tokens_across_devices=True
    # collapses the loss tensor to a Python int -> "'int' object has no attribute 'mean'".
    # Disabling it is the documented fix (unsloth #3716 / #3769).
    average_tokens_across_devices = False,
)

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    dataset_text_field = "text", max_seq_length = max_seq_len,
    packing = False, args = args,
)

# Mask the prompt: compute loss only on the assistant's JSON (completion). The
# marker strings are chat-template-specific and DIFFER by model family — Phi-4
# (14B) uses <|im_start|>...<|im_sep|>, but Phi-4-*mini* uses <|user|>/<|assistant|>.
# So we DERIVE the exact markers from the tokenizer's own template with sentinel
# content, guaranteeing they match `to_text` verbatim for any base model.
if SFT.get("train_on_responses_only", True):
    _U, _A = "USER_CONTENT", "ASSISTANT_CONTENT"   # sentinels
    _probe = tokenizer.apply_chat_template(
        [{"role": "user", "content": _U}, {"role": "assistant", "content": _A}],
        tokenize = False,
    )
    instruction_part = _probe.split(_U)[0]                       # prefix up to+incl. the user marker
    response_part = _probe.split(_U)[1].split(_A)[0]             # text between user content and assistant content
    print("derived instruction_part:", repr(instruction_part))
    print("derived response_part   :", repr(response_part))
    assert response_part and response_part in ds[0]["text"], (
        f"response marker {response_part!r} not found in formatted text — "
        "chat template mismatch; do not train (loss would be on nothing)."
    )
    trainer = train_on_responses_only(
        trainer,
        instruction_part = instruction_part,
        response_part = response_part,
    )

# Sanity: with max_seq_len=4096 essentially NO example should be dropped for
# truncation. A large drop here means seq len is too small again.
n_kept = len(trainer.train_dataset)
print(f"training on {n_kept} examples (dropped {len(ds) - n_kept} to truncation)")

stats = trainer.train()
print(stats)

# --- Save the LoRA adapter for download. The GGUF export runs in a SEPARATE,
# proven export-only kernel (tensor-level merge + tokenizer_class=GPT2Tokenizer),
# so we deliberately do NOT export GGUF here. ---
from pathlib import Path
ADAPTER_OUT = Path("/kaggle/working/adapter")
model.save_pretrained(str(ADAPTER_OUT))
tokenizer.save_pretrained(str(ADAPTER_OUT))
print("saved adapter ->", ADAPTER_OUT, sorted(p.name for p in ADAPTER_OUT.iterdir()))

## 5 · Sanity generation (held-in example)

In [ ]:
FastLanguageModel.for_inference(model)
sample = raw[0]
msgs = [{"role": "user", "content": sample["prompt"]}]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                       return_tensors="pt").to(model.device)
out = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.0)
print("GOLD:", sample["completion"])
print("PRED:", tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## 6 · Save + verify the LoRA adapter

This kernel **trains only** and outputs the LoRA adapter to `/kaggle/working/adapter/`.
GGUF export is done afterwards by the dedicated **export-only** kernel
(`03b_export_gguf`) — a tensor-level merge that sets `tokenizer_class=GPT2Tokenizer`
so llama.cpp takes Phi-4-mini's BPE path (the in-notebook `save_pretrained_gguf`
route trips the SentencePiece `tokenizer.model` check). The cell below confirms the
fused `qkv_proj`/`gate_up_proj` modules actually attached this run.

In [ ]:
# Verify the adapter adapted the FUSED modules this time. Run 1's weak gain came
# from ONLY o_proj + down_proj attaching — q/k/v/gate/up never matched because
# Phi-3/Phi-4-mini fuse them into qkv_proj + gate_up_proj. We now target the fused
# names, so all attention + MLP projections should carry LoRA.
from safetensors import safe_open
import glob, re, collections
f = glob.glob("/kaggle/working/adapter/adapter_model.safetensors")[0]
mods = collections.Counter()
with safe_open(f, "pt") as h:
    for k in h.keys():
        m = re.search(r"layers\.\d+\.(.*?)\.lora_[AB]", k)
        if m:
            mods[m.group(1)] += 1
print("adapted module types:", dict(mods))
print("target_modules (config):", LORA.get("target_modules"))
assert any("qkv" in m for m in mods) and any("gate_up" in m for m in mods), \
    "fused qkv/gate_up did NOT attach — check target_modules in config/train.yaml"
print("OK: fused attention + MLP modules adapted. Download /kaggle/working/adapter/.")

## 7 · Next steps

1. Download `/kaggle/working/adapter/` (the trained LoRA adapter).
2. Run the **export-only** kernel (`03b_export_gguf`) on it → `phi4-mini-ft.Q4_K_M.gguf`.
3. `ollama create phi4-mini-ft -f Modelfile` (FROM the gguf + base chat template).
4. Remove any prior `phi4-mini-ft` rows from `results/runs.jsonl`, then
   `python scripts/run_eval.py --models phi4-mini-ft` → 80 cells (baseline untouched).
5. `python scripts/make_plots.py` → before→after delta in `results/REPORT.md`.

SFT data is model-agnostic — to add mistral-7b, uncomment its block in
`config/train.yaml` + `config/config.yaml` and re-run.